In [ ]:
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130
# !pip install peft

Looking in indexes: https://download.pytorch.org/whl/cu130


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
max_length = 128
output_dir = "./my_qwen_lora"

In [4]:
raw_data = load_dataset("json", data_files="data/train.jsonl")
raw_data

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 25
    })
})

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [6]:
def format_example(sample):
    instruction = sample.get("instruction", "")
    input_text = sample.get("input", "")
    output_text = sample.get("output", "")

    if input_text.strip():
        text = f"Instruction: {instruction}\nInput: {input_text}\nResponse: {output_text}"
    else:
        text = f"Instruction: {instruction}\nResponse: {output_text}"
    return {"text": text}

formatted_data = raw_data["train"].map(format_example)
formatted_data[0]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

{'instruction': 'List the main obligations created by Article 10 of the NIS2 Directive as bullet points.',
 'input': 'Title: Article 10 - Computer security incident response teams (CSIRTs)\nArticle: 10\n\nText:\n### Article 10 - Computer security incident response teams (CSIRTs)\nEach Member State shall designate or establish one or more CSIRTs. The CSIRTs may be designated or established within a competent authority. The CSIRTs shall comply with the requirements set out in',
 'output': '- Article 10 - Computer security incident response teams (CSIRTs) Each Member State shall designate or establish one or more CSIRTs\n- The CSIRTs may be designated or established within a competent authority\n- The CSIRTs shall comply with the requirements set out in',
 'text': 'Instruction: List the main obligations created by Article 10 of the NIS2 Directive as bullet points.\nInput: Title: Article 10 - Computer security incident response teams (CSIRTs)\nArticle: 10\n\nText:\n### Article 10 - Compute

In [7]:
def preprocess(sample):
    tokenized = tokenizer(
        sample["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_data = formatted_data.map(
    preprocess,
    remove_columns=formatted_data.column_names
)

tokenized_data[0]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

{'input_ids': [16664,
  25,
  1759,
  279,
  1887,
  29155,
  3465,
  553,
  13355,
  220,
  16,
  15,
  315,
  279,
  451,
  1637,
  17,
  56752,
  438,
  17432,
  3501,
  624,
  2505,
  25,
  10869,
  25,
  13355,
  220,
  16,
  15,
  481,
  17407,
  4763,
  10455,
  2033,
  7263,
  320,
  6412,
  30521,
  82,
  340,
  16651,
  25,
  220,
  16,
  15,
  271,
  1178,
  510,
  14374,
  13355,
  220,
  16,
  15,
  481,
  17407,
  4763,
  10455,
  2033,
  7263,
  320,
  6412,
  30521,
  82,
  340,
  4854,
  12039,
  3234,
  4880,
  74124,
  476,
  5695,
  825,
  476,
  803,
  10006,
  30521,
  82,
  13,
  576,
  10006,
  30521,
  82,
  1231,
  387,
  23195,
  476,
  9555,
  2878,
  264,
  39783,
  11198,
  13,
  576,
  10006,
  30521,
  82,
  4880,
  25017,
  448,
  279,
  8502,
  738,
  700,
  304,
  198,
  2582,
  25,
  481,
  13355,
  220,
  16,
  15,
  481,
  17407,
  4763,
  10455,
  2033,
  7263,
  320,
  6412,
  30521,
  82,
  8,
  8886,
  12039,
  3234,
  4880],
 'attention_mask':

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

c:\Users\miron\anaconda3\Lib\site-packages\accelerate\utils\modeling.py:804: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [12]:
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator,
)

torch.cuda.empty_cache()
trainer.train()

Step,Training Loss
10,2.424230


TrainOutput(global_step=12, training_loss=2.376036604245504, metrics={'train_runtime': 38.611, 'train_samples_per_second': 1.942, 'train_steps_per_second': 0.311, 'total_flos': 71986454369280.0, 'train_loss': 2.376036604245504, 'epoch': 3.0})

In [13]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

('./my_qwen_lora\\tokenizer_config.json',
 './my_qwen_lora\\chat_template.jinja',
 './my_qwen_lora\\tokenizer.json')

In [14]:
from peft import PeftModel, PeftConfig

path = output_dir

config = PeftConfig.from_pretrained(path)

base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

inference_model = PeftModel.from_pretrained(base_model, path)
inference_tokenizer = AutoTokenizer.from_pretrained(path, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [15]:
prompt = "Summarize Article 21 of NIS2"
inputs = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)

with torch.no_grad():
    output = inference_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=300,
        do_sample=False,
    )

print(inference_tokenizer.decode(output[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Summarize Article 21 of NIS2

Article 21 of the Network and Information Systems (NIS) Directive is a key part of the European Union's cybersecurity strategy. It sets out requirements for critical infrastructure operators to implement security measures in order to protect their networks, systems, and data from cyber threats.

The article requires that critical infrastructure operators must have an incident response plan in place, which includes procedures for detecting, analyzing, responding to, and recovering from incidents. The plan should also include regular testing and training exercises to ensure that staff are familiar with the plan and can respond effectively in case of an actual incident.

In addition, Article 21 requires critical infrastructure operators to conduct risk assessments on a regular basis to identify potential vulnerabilities and weaknesses in their network or system. This assessment should be carried out by independent third parties who will provide a report detai